# Notebook 02 — Foveation, R-Blur and Trace-Based Metameric Periphery

**Ablations:** A1–A6 (periphery regime sweep), E1–E3 (blur curriculum)  
**Inputs:** CIFAR-10 (smoke test) · ImageNet-100/1K (full run)  
**Outputs:** `src/foveation.py` · `results/02_periphery_samples.png` · `results/02_snr_profile.png` · `results/02_metrics.json`

---

## Purpose

This notebook implements and ablates the **foveation component** (architecture
components C1 and C2) of the thesis model:

1. **R-Blur foveal transform** — eccentricity-dependent Gaussian blur creating a sharp
   fovea and progressively blurred periphery.
2. **Three periphery regimes** compared at equal average SNR budget:
   - **A2 — Flat blur:** low-pass Gaussian only (no noise term).
   - **A3 — i.i.d. noise:** signal-independent Gaussian noise (like VOneNet Poisson noise).
   - **A4 — Trace-based metameric (original contribution):** signal-conditioned AdaIN
     noise whose local statistics match the original image — the *trace* of the signal
     is preserved while fine phase structure is randomised.
3. **Mandatory visualisation** of each regime *before any training*, with measured
   $\mathrm{SNR}(r)$ curves.
4. **Ablation smoke test:** tiny classifier on CIFAR-10, one pass per periphery mode.

## Mathematics covered

$$
r = \lVert \mathbf p - \mathbf p_0 \rVert \quad \text{(eccentricity, pixels from fixation)}
$$

$$
60\,s(r) \approx 0.53 + 0.434\,r \quad\Rightarrow\quad
s(r) = \frac{0.53 + 0.434\,r}{60}\;[\text{deg}]
\quad\text{[Watson 2014]}
$$

$$
\mathrm{SNR}(r) = \mathrm{SNR}_0 \left(\frac{s_0}{s(r)}\right)^{\!\beta},
\qquad
\mathrm{SNR}_{\mathrm{dB}}(r) = 10\log_{10}\mathrm{SNR}(r)
$$

$$
\alpha(r) = \frac{\sqrt{\mathrm{SNR}(r)/\rho}}{1 + \sqrt{\mathrm{SNR}(r)/\rho}},
\qquad
\hat{I}_\alpha = \alpha(r)\,I + (1-\alpha(r))\,T(S_{\Omega(r)}(I))
$$


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys, os, warnings, importlib
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    # Walk upward from CWD to find the project root (contains src/)
    _here = os.getcwd()
    for _d in [_here] + [os.path.dirname(_here)] + [os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d
            break
    else:
        PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# ── Pip install (Colab / fresh env only) ───────────────────────────────────
# !pip -q install torch torchvision timm scipy scikit-learn matplotlib tqdm

# ── Imports ─────────────────────────────────────────────────────────────────
import math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for nbclient/Colab
import matplotlib.pyplot as plt
from PIL import Image

from src import common, datasets

# ── Configuration ──────────────────────────────────────────────────────────
CFG = {
    # General
    "seed":           42,
    "smoke_test":     True,      # False → full run (real CIFAR-10 data, more epochs)
    "dataset":        "cifar10", # "cifar10" | "imagenet" -- geometry demo default only;
                                  # the ablation loop below sweeps CFG["eval_datasets"]
    # Image geometry
    "image_size":     32,
    "ppd":            4.0,       # 32 px / 8° = 4 ppd for CIFAR-10
    "fixation_yx":    (0.5, 0.5),
    # R-Blur
    "rblur_sigma0":   0.5,       # blur sigma at fovea [px]
    "rblur_slope":    1.5,       # px of blur added per degree of eccentricity
    # Foveal mask
    "fovea_deg":      1.0,       # fovea radius [deg]
    "transition_deg": 0.5,       # soft-mask transition width [deg]
    # SNR profile
    "snr0_db":        30.0,      # foveal SNR [dB]
    "beta":           2.0,       # SNR roll-off exponent (2 = area-based pooling)
    "beta_sweep":     [1.0, 2.0, 3.0, 5.0, 8.0],   # widened: 1-3 barely change alpha(r) within CIFAR-10's 8deg field of view; 5,8 show the effect clearly (see chat discussion)
    "rho":            1.0,       # signal/texture power ratio for alpha(r)
    # Trace-based periphery
    "patch_size":     8,         # AdaIN patch size [px] (scales with image_size)
    # Smoke-test training
    "n_epochs_smoke": 3,
    "n_batches_smoke": 30,
    "batch_size":     64,
    "lr":             1e-3,
    # Full-run training (smoke_test=False)
    "n_epochs_full":  20,        # per (dataset, ablation mode) condition
    "n_batches_full": None,      # None = full dataset each epoch
    # Ablation modes to evaluate
    "ablation_modes": ["none", "blur", "iid", "trace"],
    # Real-data ablation sweep (per docs/experiment-plan-and-ablations.md §4):
    # CIFAR-10 is the mandatory P1 fast-iteration pass; ImageNet-100 is the
    # FoveaTer-style efficiency-comparison pass for this notebook's Phase 2.
    # ImageNet-100 is license-gated / not auto-downloaded -- if it isn't present
    # under data/imagenet100/{train,val}/<class>/*.JPEG, the loop below skips it
    # with a clear message (honest fallback, same policy as `datasets.get_imagenet100`).
    "eval_datasets":  ["cifar10", "imagenet100"],
    # Paths
    "result_dir":  os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":    os.path.join(PROJECT_ROOT, "checkpoints"),
    "data_dir":    os.path.join(PROJECT_ROOT, "data"),
}

if CFG["dataset"] == "imagenet":
    CFG.update(image_size=224, ppd=28.0, fovea_deg=1.5,
               rblur_slope=2.0, patch_size=32)

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
os.makedirs(CFG["ckpt_dir"],   exist_ok=True)

print(f"Device : {device}")
print(f"Dataset: {CFG['dataset']}  |  image_size: {CFG['image_size']}  |  ppd: {CFG['ppd']}")
print(f"Smoke-test mode: {CFG['smoke_test']}")

## Math 1: Eccentricity Map and R-Blur Foveal Transform

### 1.1 Eccentricity map

Let $\mathbf p = (p_y, p_x)$ be a pixel coordinate and $\mathbf p_0 = (c_y, c_x)$ the
fixation point. Pixel eccentricity (distance from fixation) in degrees:

$$
r(\mathbf p) = \frac{1}{\delta}\,\lVert \mathbf p - \mathbf p_0 \rVert_2, \qquad
\delta = \frac{\text{image size [px]}}{\text{visual angle [°]}} \;\;[\text{ppd}]
$$

### 1.2 R-Blur: eccentricity-dependent Gaussian blur

At each pixel $\mathbf p$, the blur scale grows linearly with eccentricity:

$$
\sigma(\mathbf p) = \sigma_0 + \kappa\,r(\mathbf p) \quad [\text{pixels}]
$$

where $\sigma_0$ is the minimum foveal blur and $\kappa$ the rate of increase.

**Implementation — Gaussian pyramid blending.** Precompute $N$ blurred images
$\{B_0, B_1, \ldots, B_{N-1}\}$ at evenly-spaced sigma levels $\sigma_0 \le \sigma_k
\le \sigma_{\max}$. At each pixel, assign triangle-window weights:

$$
w_k(\mathbf p) = \max\!\left(0,\; 1 - \frac{|\sigma(\mathbf p) - \sigma_k|}
{\Delta\sigma}\right),
\qquad \hat w_k = \frac{w_k}{\sum_j w_j},
\qquad
\mathrm{RBlur}(I)(\mathbf p) = \sum_k \hat w_k(\mathbf p)\, B_k(\mathbf p).
$$

This is equivalent to linear interpolation between adjacent blur levels and is
differentiable almost everywhere.

### 1.3 Composite foveated image

A soft foveal mask $m(r)$ separates the sharp fovea from the periphery:

$$
m(r) = \sigma\!\left(\frac{r_{\text{fovea}} - r}{\tau}\right), \qquad
I_{\text{fov}} = m(r)\,I + (1 - m(r))\,\mathrm{Periph}(I)
$$

where $\sigma(\cdot)$ is the logistic sigmoid, $r_{\text{fovea}}$ is the fovea radius,
and $\tau$ is the transition width.


In [ ]:
# ── Core geometry helpers (inline — mirrored in src/foveation.py) ───────────

def eccentricity_map(H, W, fixation_yx=(0.5, 0.5), ppd=28.0):
    """Eccentricity map for an HxW image.
    Returns r_pix [H,W] in pixels and r_deg [H,W] in degrees."""
    cy, cx = fixation_yx[0] * H, fixation_yx[1] * W
    ys = torch.arange(H, dtype=torch.float32)
    xs = torch.arange(W, dtype=torch.float32)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    r_pix = torch.sqrt((gy - cy) ** 2 + (gx - cx) ** 2)
    r_deg = r_pix / ppd
    return r_pix, r_deg


def gaussian_blur_2d(image, sigma):
    """Isotropic Gaussian blur on [C,H,W] or [B,C,H,W]. Returns unchanged if sigma<1e-3."""
    if sigma < 1e-3:
        return image
    squeeze = image.dim() == 3
    if squeeze:
        image = image.unsqueeze(0)
    B, C, H, W = image.shape
    radius = max(1, int(math.ceil(3.0 * sigma)))
    ks = 2 * radius + 1
    xs = torch.arange(-radius, radius + 1, dtype=image.dtype, device=image.device)
    k1d = torch.exp(-0.5 * (xs / sigma) ** 2)
    k1d = k1d / k1d.sum()
    k2d = (k1d[:, None] * k1d[None, :]).view(1, 1, ks, ks).expand(C, 1, ks, ks).contiguous()
    blurred = F.conv2d(F.pad(image, [radius]*4, mode="reflect"), k2d, groups=C)
    return blurred.squeeze(0) if squeeze else blurred


def soft_foveal_mask(H, W, fixation_yx=(0.5, 0.5), fovea_deg=1.5,
                      transition_deg=0.5, ppd=28.0):
    """Sigmoid soft foveal mask: ~1 inside fovea, ~0 in periphery."""
    _, r_deg = eccentricity_map(H, W, fixation_yx, ppd)
    return torch.sigmoid(-(r_deg - fovea_deg) / transition_deg)


def apply_rblur(image, fixation_yx=(0.5, 0.5), sigma0=0.5, slope=1.5,
                ppd=28.0, n_levels=5):
    """R-Blur: eccentricity-dependent Gaussian blur via Gaussian pyramid blending.
    image: [C,H,W] float in [0,1].  Returns [C,H,W]."""
    C, H, W = image.shape
    device = image.device
    _, r_deg = eccentricity_map(H, W, fixation_yx, ppd)
    r_deg = r_deg.to(device)
    sigma_map = (sigma0 + slope * r_deg).clamp(min=0.0)
    sigma_max = float(sigma_map.max().item())
    sigma_levels = torch.linspace(0.0, sigma_max, n_levels)
    blurred_stack = torch.stack(
        [gaussian_blur_2d(image, float(s.item())) for s in sigma_levels], dim=0
    )  # [N, C, H, W]
    level_span = (sigma_levels[1] - sigma_levels[0]).clamp(min=1e-8)
    dist = (sigma_map[None] - sigma_levels[:, None, None]).abs() / level_span
    weights = F.relu(1.0 - dist)
    weights = weights / (weights.sum(0, keepdim=True) + 1e-8)
    return (weights[:, None] * blurred_stack).sum(0).clamp(0.0, 1.0)


# ── Unit tests ─────────────────────────────────────────────────────────────
S = CFG["image_size"]
ppd = CFG["ppd"]

r_pix, r_deg = eccentricity_map(S, S, CFG["fixation_yx"], ppd)
assert r_deg[S//2, S//2] < 0.05, "Fixation pixel should have near-zero eccentricity"
max_r = float(r_deg.max().item())
print(f"Eccentricity range: [{r_deg.min():.3f}, {max_r:.3f}] deg  "
      f"(corner = {float(r_deg[0,0].item()):.2f} deg)")

mask = soft_foveal_mask(S, S, CFG["fixation_yx"], CFG["fovea_deg"],
                         CFG["transition_deg"], ppd)
print(f"Foveal mask: centre={mask[S//2,S//2]:.3f}  corner={mask[0,0]:.3f}")

dummy = torch.rand(3, S, S)
rblur = apply_rblur(dummy, CFG["fixation_yx"], CFG["rblur_sigma0"],
                    CFG["rblur_slope"], ppd)
assert rblur.shape == dummy.shape, "R-Blur must preserve shape"
assert rblur.min() >= 0.0 and rblur.max() <= 1.0, "R-Blur must stay in [0,1]"
diff_centre = (rblur[:, S//2, S//2] - dummy[:, S//2, S//2]).abs().mean().item()
diff_corner = (rblur[:, 0, 0] - dummy[:, 0, 0]).abs().mean().item()
print(f"R-Blur pixel change — centre: {diff_centre:.4f}  corner: {diff_corner:.4f}")
print("✓ Core geometry helpers OK")


## Math 2: Watson (2014) mRGC Pooling Scale and SNR Profile

### 2.1 Watson pooling scale

Watson (2014, *J. Vision* 14(7):15) derives the mean midget-RGC receptive-field
spacing $s(r)$ from Curcio & Allen (1990) density measurements:

$$
60\,s(r) \approx 0.53 + 0.434\,r, \quad r\ \text{[degrees]}
\quad\Rightarrow\quad
\boxed{s(r) = \frac{0.53 + 0.434\,r}{60}\ \text{[deg]}}
$$

Key values:
- $s(0) = 0.53/60 \approx 0.0088°$ ≈ 0.53 arcmin (cone spacing at fovea).
- $s(r)$ grows nearly linearly with eccentricity → peripheral pooling area $\propto s(r)^2$.

### 2.2 SNR profile (thesis original)

Peripheral pooling over area $A(r) \propto w(r)^2 \propto s(r)^2$ reduces fine-detail
fidelity. The resulting local SNR obeys a power-law roll-off
(TRACE_BASED_NOISE_REHBERI.md Sec. 3.2):

$$
\boxed{\mathrm{SNR}(r) = \mathrm{SNR}_0\left(\frac{s(0)}{s(r)}\right)^{\!\beta}
= \mathrm{SNR}_0\left(\frac{0.53}{0.53 + 0.434\,r}\right)^{\!\beta}}
$$

- $\beta = 2$: area-based pooling (default, analytically motivated).
- $\beta$ is an ablation knob: $\beta=1$ (slow roll-off) vs. $\beta=3$ (steep).
- $\mathrm{SNR}_0$: foveal reference (set to 30 dB = $10^3$).

### 2.3 Content–texture interpolation coefficient

Given target $\mathrm{SNR}(r)$, the content fraction $\alpha(r) \in [0,1]$ satisfies
(Sec. 3.3):

$$
\hat I_\alpha = \alpha(r)\,I + (1-\alpha(r))\,T(S_{\Omega}(I)),
\quad
\mathrm{SNR}(\alpha) \approx \frac{\alpha^2}{(1-\alpha)^2}\,\rho
\quad\Rightarrow\quad
\boxed{\alpha(r) = \frac{\sqrt{\mathrm{SNR}(r)/\rho}}{1 + \sqrt{\mathrm{SNR}(r)/\rho}}}
$$


In [ ]:
# ── WatsonPoolingScale ─────────────────────────────────────────────────────

class WatsonPoolingScale:
    """Watson (2014) mRGC spacing: s(r) = (0.53 + 0.434*r) / 60 [deg]."""
    A, B = 0.434, 0.53   # slope [arcmin/deg], intercept [arcmin]

    def spacing(self, r_deg):
        """s(r) in degrees."""
        return (self.B + self.A * r_deg) / 60.0

    def spacing_fovea(self):
        """s(0) in degrees."""
        return self.B / 60.0

    def window_radius(self, r_deg, kappa=1.0):
        """w(r) = kappa * s(r) [deg]."""
        return kappa * self.spacing(r_deg)


# ── SNRProfile ─────────────────────────────────────────────────────────────

class SNRProfile:
    """SNR(r) = SNR0 * (s(0)/s(r))^beta, with alpha(r) interpolation knob."""

    def __init__(self, snr0_db=30.0, beta=2.0, ppd=28.0, rho=1.0):
        self.snr0 = 10.0 ** (snr0_db / 10.0)
        self.beta = beta
        self.ppd = ppd
        self.rho = rho
        self._watson = WatsonPoolingScale()

    def snr(self, r_deg):
        """Linear SNR(r). r_deg: float or Tensor."""
        s0 = self._watson.spacing_fovea()
        sr = self._watson.spacing(r_deg)
        return self.snr0 * (s0 / sr) ** self.beta

    def snr_db(self, r_deg):
        """SNR in dB."""
        return 10.0 * torch.log10(self.snr(r_deg))

    def alpha(self, r_deg):
        """Content-texture interpolation alpha(r) in [0,1]."""
        snr_r = self.snr(r_deg)
        sq = torch.sqrt(snr_r / self.rho)
        return sq / (1.0 + sq)


# ── Numerical validation ────────────────────────────────────────────────────

watson = WatsonPoolingScale()

# Closed-form checks from Watson (2014)
s0 = watson.spacing_fovea()
print(f"s(0) = {s0:.6f} deg = {s0*60:.3f} arcmin  [expected 0.53 arcmin]")
assert abs(s0 * 60 - 0.53) < 0.001, "Watson intercept mismatch"

s5 = watson.spacing(5.0)
print(f"s(5°) = {s5:.5f} deg = {s5*60:.3f} arcmin")

# SNR profile
prof = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                   ppd=CFG["ppd"], rho=CFG["rho"])
r_test = torch.tensor([0.001, 1.0, 3.0, 5.0, 7.0])
snr_test = prof.snr(r_test)
alpha_test = prof.alpha(r_test)

print("\nSNR profile validation:")
print(f"{'r [deg]':>8}  {'SNR_lin':>12}  {'SNR_dB':>8}  {'alpha':>7}")
for r, s, a in zip(r_test.tolist(), snr_test.tolist(), alpha_test.tolist()):
    print(f"{r:8.3f}  {s:12.1f}  {10*np.log10(s):8.2f}  {a:7.4f}")

# Monotonicity check
assert (snr_test[1:] < snr_test[:-1]).all(), "SNR must decrease with eccentricity"
assert (alpha_test[1:] < alpha_test[:-1]).all(), "alpha must decrease with eccentricity"
# Alpha bounds
assert alpha_test.min() > 0.0 and alpha_test.max() < 1.0, "alpha must be in (0,1)"
print("\n✓ WatsonPoolingScale and SNRProfile validation passed")


## Math 3: Three Periphery Regimes at Equal SNR Budget

The ablation (A2–A4) compares three periphery models that share the **same mean SNR
budget** but differ in their noise *structure*:

| Ablation | Regime | Signal model | Trace preserved? |
|----------|--------|--------------|-----------------|
| A2 | **Flat blur** | $\hat I = I * G_{\sigma(r)}$ — low-pass only | Partially (low-freq.) |
| A3 | **i.i.d. noise** | $\hat I = I + \eta,\; \eta \sim \mathcal N(0,\sigma^2_{\text{SNR}})$ | No |
| A4 | **Trace-based metameric** | $\hat I = \alpha(r)\,I + (1-\alpha(r))\,T(S_\Omega(I))$ | **Yes** |

### A2 — Flat blur

$$
\hat I(\mathbf p) = (I * G_{\sigma(r)})(\mathbf p),
\qquad \sigma(r) = \sigma_0 + \kappa\,r
$$

No additive noise term. High-frequency signal power is *attenuated*, not replaced.

### A3 — i.i.d. noise

$$
\hat I(\mathbf p) = I(\mathbf p) + \eta(\mathbf p),
\quad \eta \sim \mathcal N\!\left(0,\;\sigma_{\eta}^2(r)\right),
\quad \sigma_{\eta}^2(r) = \frac{\mathrm{Var}[I]}{\mathrm{SNR}(r)}
$$

Noise is drawn i.i.d., independent of the image content. No local texture statistics.

### A4 — Trace-based metameric (original)

$$
\hat I(\mathbf p) = \alpha(r)\,I(\mathbf p) + (1 - \alpha(r))\,\underbrace{T\bigl(S_{\Omega(r)}(I)\bigr)(\mathbf p)}_{\text{AdaIN texture sample}}
$$

Local statistics $S_\Omega = \{\mu_\Omega, \sigma_\Omega\}$ (mean and std over a patch of
radius $\propto w(r)$) define the **trace**. The texture synthesiser $T$:

$$
T(S_\Omega(I))(\mathbf p) = \sigma_\Omega \cdot
\frac{\xi(\mathbf p) - \mu_\xi}{\sigma_\xi} + \mu_\Omega,
\qquad \xi \sim \mathcal N(0, I)
$$

(AdaIN: normalise Gaussian noise, then rescale/shift to match patch $(\mu, \sigma)$).

**Equal-SNR calibration.** All three regimes are compared after setting their free
parameters so that the mean SNR (averaged over eccentricity bins) is equal.


In [ ]:
# ── Periphery regime modules ──────────────────────────────────────────────

class FlatBlurPeriphery(nn.Module):
    """A2: Eccentricity-dependent Gaussian blur (R-Blur, no added noise)."""
    def __init__(self, sigma0=0.5, slope=2.0, ppd=28.0, fixation_yx=(0.5, 0.5)):
        super().__init__()
        self.sigma0 = sigma0; self.slope = slope
        self.ppd = ppd; self.fixation_yx = fixation_yx

    def forward(self, image):
        return apply_rblur(image, self.fixation_yx, self.sigma0, self.slope, self.ppd)


class IIDNoisePeriphery(nn.Module):
    """A3: Signal-independent i.i.d. Gaussian noise calibrated to target SNR(r)."""
    def __init__(self, snr_profile, fixation_yx=(0.5, 0.5), ppd=28.0):
        super().__init__()
        self.snr_profile = snr_profile
        self.fixation_yx = fixation_yx; self.ppd = ppd

    def forward(self, image):
        C, H, W = image.shape
        device = image.device
        _, r_deg = eccentricity_map(H, W, self.fixation_yx, self.ppd)
        r_deg = r_deg.to(device)
        snr_map = self.snr_profile.snr(r_deg).clamp(min=1e-3)    # [H, W]
        signal_var = image.var().clamp(min=1e-6)
        sigma_map = torch.sqrt(signal_var / snr_map)              # [H, W]
        noise = torch.randn_like(image)
        return (image + sigma_map[None] * noise).clamp(0.0, 1.0)


class TraceBasedPeriphery(nn.Module):
    """A4 (original): Signal-conditioned AdaIN metameric periphery noise.

    hat_I(p) = alpha(r) * I(p) + (1-alpha(r)) * T(S_{Omega(r)}(I))(p)
    T(S) = AdaIN(Gaussian_noise, local_stats)  — preserves mean+std, randomises phase.
    """
    def __init__(self, snr_profile, patch_size=8, fixation_yx=(0.5, 0.5), ppd=28.0):
        super().__init__()
        self.snr_profile = snr_profile
        self.patch_size = patch_size
        self.fixation_yx = fixation_yx; self.ppd = ppd

    def forward(self, image):
        C, H, W = image.shape
        device = image.device
        _, r_deg = eccentricity_map(H, W, self.fixation_yx, self.ppd)
        r_deg = r_deg.to(device)
        alpha_map = self.snr_profile.alpha(r_deg)                 # [H, W]
        texture = self._adain_texture(image)                      # [C, H, W]
        return (alpha_map[None] * image + (1.0 - alpha_map[None]) * texture
                ).clamp(0.0, 1.0)

    def _adain_texture(self, image):
        """AdaIN patch texture: Gaussian noise → match local (mean, std) per patch."""
        C, H, W = image.shape
        device = image.device
        p = self.patch_size
        pad_h = (p - H % p) % p
        pad_w = (p - W % p) % p
        img_p = F.pad(image, [0, pad_w, 0, pad_h], mode="reflect")  # [C, Hp, Wp]
        noise_p = torch.randn(C, H + pad_h, W + pad_w, device=device, dtype=image.dtype)
        # [C, nH, nW, p, p]
        img_unf   = img_p.unfold(1, p, p).unfold(2, p, p)
        noise_unf = noise_p.unfold(1, p, p).unfold(2, p, p)
        mu    = img_unf.mean(dim=(-2, -1), keepdim=True)
        sigma = img_unf.std(dim=(-2, -1), keepdim=True).clamp(min=1e-5)
        n_mu  = noise_unf.mean(dim=(-2, -1), keepdim=True)
        n_std = noise_unf.std(dim=(-2, -1), keepdim=True).clamp(min=1e-5)
        adain = sigma * (noise_unf - n_mu) / n_std + mu
        nH = (H + pad_h) // p
        nW = (W + pad_w) // p
        texture = adain.permute(0, 1, 3, 2, 4).reshape(C, nH * p, nW * p)
        return texture[:, :H, :W].clamp(0.0, 1.0)


# ── Quick forward-pass smoke tests ─────────────────────────────────────────
_snr_prof = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                        ppd=CFG["ppd"], rho=CFG["rho"])
_img = torch.rand(3, CFG["image_size"], CFG["image_size"])

_blur_mod  = FlatBlurPeriphery(CFG["rblur_sigma0"], CFG["rblur_slope"],
                                CFG["ppd"], CFG["fixation_yx"])
_iid_mod   = IIDNoisePeriphery(_snr_prof, CFG["fixation_yx"], CFG["ppd"])
_trace_mod = TraceBasedPeriphery(_snr_prof, CFG["patch_size"],
                                  CFG["fixation_yx"], CFG["ppd"])

for name, mod in [("FlatBlur", _blur_mod), ("IIDNoise", _iid_mod), ("Trace", _trace_mod)]:
    out = mod(_img)
    assert out.shape == _img.shape, f"{name}: shape mismatch"
    assert 0.0 <= out.min() and out.max() <= 1.0, f"{name}: output out of [0,1]"
    diff = (out - _img).abs().mean().item()
    print(f"{name:10s}  diff_mean={diff:.4f}  range=[{out.min():.3f},{out.max():.3f}]")

print("✓ All periphery modules forward-pass OK")


In [ ]:
# ── FoveatedTransform: composite fovea + periphery ───────────────────────

class FoveatedTransform(nn.Module):
    """Composite: sharp fovea mask + chosen periphery module.

    Output = m(r) * I + (1-m(r)) * Periphery(I)
    m(r): soft sigmoid mask (~1 inside fovea, ~0 in periphery).
    periphery=None → A1 (no-op, identity transform).
    """
    def __init__(self, periphery=None, fovea_deg=1.5, transition_deg=0.5,
                 ppd=28.0, fixation_yx=(0.5, 0.5)):
        super().__init__()
        self.periphery = periphery
        self.fovea_deg = fovea_deg; self.transition_deg = transition_deg
        self.ppd = ppd; self.fixation_yx = fixation_yx

    def forward(self, image):
        if self.periphery is None:
            return image
        C, H, W = image.shape
        device = image.device
        mask = soft_foveal_mask(H, W, self.fixation_yx, self.fovea_deg,
                                self.transition_deg, self.ppd).to(device)  # [H,W]
        periph = self.periphery(image)
        return (mask[None] * image + (1.0 - mask[None]) * periph).clamp(0.0, 1.0)


def build_foveated_transform(mode, snr_profile,
                              patch_size=8, fovea_deg=1.5, transition_deg=0.5,
                              ppd=28.0, fixation_yx=(0.5, 0.5),
                              blur_sigma0=0.5, blur_slope=2.0):
    """Factory: build FoveatedTransform for mode in {none,blur,iid,trace}."""
    if mode == "none":
        periph = None
    elif mode in ("blur", "rblur"):
        periph = FlatBlurPeriphery(blur_sigma0, blur_slope, ppd, fixation_yx)
    elif mode == "iid":
        periph = IIDNoisePeriphery(snr_profile, fixation_yx, ppd)
    elif mode == "trace":
        periph = TraceBasedPeriphery(snr_profile, patch_size, fixation_yx, ppd)
    else:
        raise ValueError(f"Unknown mode: {mode!r}")
    return FoveatedTransform(periph, fovea_deg, transition_deg, ppd, fixation_yx)


# ── Integration test: FoveatedTransform preserves fovea ──────────────────
_snr_prof2 = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                         ppd=CFG["ppd"], rho=CFG["rho"])

for mode in CFG["ablation_modes"]:
    ft = build_foveated_transform(
        mode, _snr_prof2,
        patch_size=CFG["patch_size"],
        fovea_deg=CFG["fovea_deg"],
        transition_deg=CFG["transition_deg"],
        ppd=CFG["ppd"],
        fixation_yx=CFG["fixation_yx"],
        blur_sigma0=CFG["rblur_sigma0"],
        blur_slope=CFG["rblur_slope"],
    )
    out = ft(_img)
    # Fovea pixels should be nearly unchanged (mask ≈ 1)
    S = CFG["image_size"]
    fovea_diff = (out[:, S//2, S//2] - _img[:, S//2, S//2]).abs().mean().item()
    print(f"mode={mode:5s}  fovea_pixel_diff={fovea_diff:.5f}  "
          f"(should be ≈0 for all modes)")

print("✓ FoveatedTransform integration test OK")


## Mandatory Visualisation: Periphery Regimes Before Training

> **Specification (SONNET5_IMPLEMENTATION_PROMPT.md):** Before any training, load ≥ 6
> samples from the dataset and show each periphery regime side-by-side:
>
> `[original] · [R-Blur] · [flat-blur periphery] · [i.i.d. noise periphery] · [trace-based metameric periphery]`
>
> with the fixation point marked and measured $\mathrm{SNR}(r)$ on each panel.
> Save to `results/02_periphery_samples.png` and `results/02_snr_profile.png`.

The visualisation serves two purposes:
1. **Sanity check:** confirm that each regime produces visually distinct and
   mathematically coherent peripheral distortion before any training is done.
2. **SNR measurement:** confirm that the mean SNR across eccentricity bins is
   comparable across regimes (equal-budget constraint).


In [ ]:
# ── Load visualisation samples ────────────────────────────────────────────
# Priority:
#   1. Real CIFAR-10 (already on disk, no download forced)
#   2. skimage built-in photographs (real photos, no download needed)
#   3. Procedural sine-wave patterns (last resort)
# Visualisation always uses real or realistic images regardless of smoke_test.

def make_synthetic_sample(H=32, W=32, seed=0):
    """Procedural sine-wave fallback — used only if all real-image sources fail."""
    rng = np.random.RandomState(seed)
    ys = np.linspace(0, 4 * np.pi, H)
    xs = np.linspace(0, 4 * np.pi, W)
    Y, X = np.meshgrid(ys, xs, indexing="ij")
    freq  = rng.uniform(0.5, 2.0, 3)
    phase = rng.uniform(0, 2 * np.pi, 3)
    angle = rng.uniform(0, np.pi, 3)
    img = np.zeros((3, H, W), dtype=np.float32)
    for c in range(3):
        Xr = X * np.cos(angle[c]) + Y * np.sin(angle[c])
        img[c] = 0.5 + 0.4 * np.sin(freq[c] * Xr + phase[c])
    return torch.from_numpy(img.clip(0, 1))


N_VIS = 6
S = CFG["image_size"]
vis_samples = []
vis_labels  = []   # human-readable label for each sample (shown in figure title)

# ── Source 1: CIFAR-10 from disk (no download) ────────────────────────────
try:
    cifar_raw = torchvision.datasets.CIFAR10(
        root=CFG["data_dir"], train=False, download=False,
        transform=T.Compose([T.ToTensor()])   # raw [0,1], no normalisation
    )
    CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                       "dog","frog","horse","ship","truck"]
    step = max(1, len(cifar_raw) // N_VIS)
    indices = list(range(0, len(cifar_raw), step))[:N_VIS]
    for idx in indices:
        img_t, label_idx = cifar_raw[idx]
        vis_samples.append(img_t.clamp(0.0, 1.0))
        vis_labels.append(f"CIFAR-10: {CIFAR10_CLASSES[label_idx]}")
    print(f"Source 1 (CIFAR-10): loaded {len(vis_samples)} real images.")
except Exception as e:
    print(f"Source 1 (CIFAR-10) not available: {e}")

# ── Source 2: skimage built-in photographs ────────────────────────────────
if len(vis_samples) < N_VIS:
    try:
        import skimage.data as _skd
        _resize = T.Compose([
            T.ToPILImage(),
            T.Resize(S),
            T.CenterCrop(S),
            T.ToTensor(),
        ])
        _skimage_samples = [
            (_skd.astronaut(),              "astronaut"),
            (_skd.chelsea(),               "chelsea (cat)"),
            (_skd.rocket(),                "rocket"),
            (_skd.coffee(),                "coffee"),
            (_skd.hubble_deep_field(),     "Hubble deep field"),
            (_skd.immunohistochemistry(),  "immunohistochemistry"),
        ]
        needed = N_VIS - len(vis_samples)
        for arr, name in _skimage_samples[:needed]:
            # skimage images are H×W×C uint8; convert to C×H×W float
            tensor = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
            vis_samples.append(_resize(tensor))
            vis_labels.append(f"skimage: {name}")
        print(f"Source 2 (skimage): added {needed} real photographs. "
              f"Total: {len(vis_samples)}")
    except Exception as e:
        print(f"Source 2 (skimage) failed: {e}")

# ── Source 3: procedural fallback ─────────────────────────────────────────
if len(vis_samples) < N_VIS:
    needed = N_VIS - len(vis_samples)
    for i in range(needed):
        vis_samples.append(make_synthetic_sample(S, S, seed=i))
        vis_labels.append(f"synthetic pattern #{i}")
    print(f"Source 3 (synthetic): added {needed} procedural patterns. "
          f"Total: {len(vis_samples)}")

print(f"\nVisualisation samples: {len(vis_samples)} x {tuple(vis_samples[0].shape)}")
for i, lbl in enumerate(vis_labels):
    print(f"  [{i}] {lbl}")


In [ ]:
# ── Generate 5-panel visualisation ───────────────────────────────────────
# Columns: original | R-Blur | flat-blur | i.i.d. noise | trace-based
# Rows:    one per sample (real CIFAR-10 or skimage photo, with label)

common.set_seed(CFG["seed"])

snr_prof_vis = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                           ppd=CFG["ppd"], rho=CFG["rho"])

rblur_full = lambda img: apply_rblur(img, CFG["fixation_yx"],
                                      CFG["rblur_sigma0"], CFG["rblur_slope"],
                                      CFG["ppd"])
blur_ft  = build_foveated_transform("blur",  snr_prof_vis, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"],
    CFG["rblur_sigma0"], CFG["rblur_slope"])
iid_ft   = build_foveated_transform("iid",   snr_prof_vis, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"])
trace_ft = build_foveated_transform("trace", snr_prof_vis, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"])

col_labels = ["Original", "R-Blur\n(full image)", "Flat blur\nperiphery",
              "i.i.d. noise\nperiphery", "Trace-based\nmetameric (orig.)"]
transforms_vis = [lambda img: img, rblur_full, blur_ft, iid_ft, trace_ft]

n_cols = len(col_labels)
n_rows = len(vis_samples)
# Extra left margin for row labels
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(3.2 * n_cols, 2.8 * n_rows),
                          squeeze=False)
fig.suptitle(
    "Periphery Regimes — Before Training\n"
    f"fixation=centre  |  ppd={CFG['ppd']}  |  "
    f"SNR₀={CFG['snr0_db']} dB  |  β={CFG['beta']}",
    fontsize=10, y=1.01,
)

fy_px = int(CFG["fixation_yx"][0] * S)
fx_px = int(CFG["fixation_yx"][1] * S)

_, r_deg_map = eccentricity_map(S, S, CFG["fixation_yx"], CFG["ppd"])
r_corner     = float(r_deg_map[0, 0].item())
snr_corner_db = 10 * np.log10(float(snr_prof_vis.snr(torch.tensor(r_corner)).item()))

for row_i, (orig, row_lbl) in enumerate(zip(vis_samples, vis_labels)):
    for col_j, (col_lbl, tfm) in enumerate(zip(col_labels, transforms_vis)):
        with torch.no_grad():
            out = tfm(orig.clone()).clamp(0.0, 1.0)

        ax = axes[row_i][col_j]
        ax.imshow(out.permute(1, 2, 0).numpy().clip(0, 1),
                  interpolation="nearest")
        # Fixation crosshair
        ax.axhline(fy_px, color="white", lw=0.8, alpha=0.8)
        ax.axvline(fx_px, color="white", lw=0.8, alpha=0.8)
        ax.plot(fx_px, fy_px, "w+", markersize=9, markeredgewidth=1.8)
        ax.axis("off")

        # Column header (top row only)
        if row_i == 0:
            ax.set_title(col_lbl, fontsize=8, pad=4)

        # SNR annotation on periphery columns (bottom row only)
        if col_j > 0 and row_i == n_rows - 1:
            ax.set_xlabel(f"SNR(corner)={snr_corner_db:.1f} dB",
                          fontsize=7, labelpad=2)

    # Row label on the leftmost panel
    axes[row_i][0].set_ylabel(row_lbl, fontsize=7, rotation=90,
                               labelpad=4, va="center")
    axes[row_i][0].axis("off")   # restore after set_ylabel flips it

plt.tight_layout()
out_path = os.path.join(CFG["result_dir"], "02_periphery_samples.png")
common.save_figure(fig, out_path, dpi=130)
plt.close(fig)
print(f"Saved: {out_path}")


In [ ]:
# ── β-sweep visualisation: how SNR roll-off steepness changes the periphery ─
#
# Figure 1 — Full 5-panel for β=3 only (same layout as the β=2 figure above)
#             → results/02_periphery_samples_beta3.png
#
# Figure 2 — Compact comparison: trace-based periphery only, β ∈ {1, 2, 3}
#             Columns = β values, rows = images  (ablation A5 illustration)
#             → results/02_periphery_beta_sweep.png
#
# SNR(r) = SNR₀ · (s₀/s(r))^β:
#   β=1 → slow roll-off (mild peripheral degradation)
#   β=2 → area-based pooling (default, analytically motivated)
#   β=3 → steep roll-off (strong peripheral metameric noise)

common.set_seed(CFG["seed"])

# ── Figure 1: full 5-panel for β=3 ────────────────────────────────────────
beta3_prof = SNRProfile(snr0_db=CFG["snr0_db"], beta=3.0,
                         ppd=CFG["ppd"], rho=CFG["rho"])

rblur_full_b3 = lambda img: apply_rblur(img, CFG["fixation_yx"],
                                          CFG["rblur_sigma0"], CFG["rblur_slope"],
                                          CFG["ppd"])
blur_ft_b3  = build_foveated_transform("blur",  beta3_prof, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"],
    CFG["rblur_sigma0"], CFG["rblur_slope"])
iid_ft_b3   = build_foveated_transform("iid",   beta3_prof, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"])
trace_ft_b3 = build_foveated_transform("trace", beta3_prof, CFG["patch_size"],
    CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"])

col_labels_b3 = ["Original", "R-Blur\n(full)", "Flat blur\nperiphery",
                  "i.i.d. noise\nperiphery", "Trace-based\nmetameric (β=3)"]
tfms_b3 = [lambda img: img, rblur_full_b3, blur_ft_b3, iid_ft_b3, trace_ft_b3]

n_cols, n_rows = len(col_labels_b3), len(vis_samples)
fig1, ax1s = plt.subplots(n_rows, n_cols,
                            figsize=(3.2 * n_cols, 2.8 * n_rows), squeeze=False)
fig1.suptitle(
    "Periphery Regimes — β=3 (steep SNR roll-off)\n"
    f"fixation=centre  |  ppd={CFG['ppd']}  |  SNR₀={CFG['snr0_db']} dB  |  β=3.0",
    fontsize=10, y=1.01,
)
_, r_deg_map = eccentricity_map(S, S, CFG["fixation_yx"], CFG["ppd"])
r_corner = float(r_deg_map[0, 0].item())
snr_b3_db = 10 * np.log10(float(beta3_prof.snr(torch.tensor(r_corner)).item()))

for row_i, (orig, row_lbl) in enumerate(zip(vis_samples, vis_labels)):
    for col_j, (lbl, tfm) in enumerate(zip(col_labels_b3, tfms_b3)):
        with torch.no_grad():
            out = tfm(orig.clone()).clamp(0.0, 1.0)
        ax = ax1s[row_i][col_j]
        ax.imshow(out.permute(1, 2, 0).numpy().clip(0, 1), interpolation="nearest")
        ax.axhline(int(CFG["fixation_yx"][0] * S), color="white", lw=0.8, alpha=0.8)
        ax.axvline(int(CFG["fixation_yx"][1] * S), color="white", lw=0.8, alpha=0.8)
        ax.plot(int(CFG["fixation_yx"][1] * S), int(CFG["fixation_yx"][0] * S),
                "w+", markersize=9, markeredgewidth=1.8)
        ax.axis("off")
        if row_i == 0:
            ax.set_title(lbl, fontsize=8, pad=4)
        if col_j > 0 and row_i == n_rows - 1:
            ax.set_xlabel(f"SNR(corner)={snr_b3_db:.1f} dB", fontsize=7, labelpad=2)
    ax1s[row_i][0].set_ylabel(row_lbl, fontsize=7, rotation=90, labelpad=4, va="center")
    ax1s[row_i][0].axis("off")

plt.tight_layout()
path_b3 = os.path.join(CFG["result_dir"], "02_periphery_samples_beta3.png")
common.save_figure(fig1, path_b3, dpi=130)
plt.close(fig1)
print(f"Saved: {path_b3}")

# ── Figure 2: β-sweep comparison (trace-based column only) ────────────────
beta_vals = CFG["beta_sweep"]   # [1.0, 2.0, 3.0]
trace_fts_sweep = []
for bv in beta_vals:
    p = SNRProfile(snr0_db=CFG["snr0_db"], beta=bv,
                   ppd=CFG["ppd"], rho=CFG["rho"])
    trace_fts_sweep.append(
        build_foveated_transform("trace", p, CFG["patch_size"],
            CFG["fovea_deg"], CFG["transition_deg"],
            CFG["ppd"], CFG["fixation_yx"])
    )

# Extra column 0: original image (reference)
n_cols2 = 1 + len(beta_vals)
fig2, ax2s = plt.subplots(n_rows, n_cols2,
                            figsize=(2.8 * n_cols2, 2.6 * n_rows), squeeze=False)
fig2.suptitle(
    "Trace-Based Metameric Periphery — β sweep  (Ablation A5)\n"
    "Columns: original | β=1 (mild) | β=2 (default) | β=3 (steep)\n"
    f"ppd={CFG['ppd']}  |  SNR₀={CFG['snr0_db']} dB  |  "
    f"fixation=centre  |  α(r) from Watson s(r)",
    fontsize=9, y=1.02,
)
col_headers2 = ["Original"] + [f"Trace-based\nβ={bv:.0f}" for bv in beta_vals]

for row_i, (orig, row_lbl) in enumerate(zip(vis_samples, vis_labels)):
    # Column 0: original
    ax = ax2s[row_i][0]
    ax.imshow(orig.permute(1, 2, 0).numpy().clip(0, 1), interpolation="nearest")
    ax.axis("off")
    ax.set_ylabel(row_lbl, fontsize=7, rotation=90, labelpad=4, va="center")
    if row_i == 0:
        ax.set_title(col_headers2[0], fontsize=8, pad=4)

    for col_j, (tfm, bv) in enumerate(zip(trace_fts_sweep, beta_vals), start=1):
        with torch.no_grad():
            out = tfm(orig.clone()).clamp(0.0, 1.0)
        snr_dbi = 10 * np.log10(float(
            SNRProfile(snr0_db=CFG["snr0_db"], beta=bv,
                       ppd=CFG["ppd"]).snr(torch.tensor(r_corner)).item()))
        ax = ax2s[row_i][col_j]
        ax.imshow(out.permute(1, 2, 0).numpy().clip(0, 1), interpolation="nearest")
        ax.axhline(int(CFG["fixation_yx"][0] * S), color="white", lw=0.8, alpha=0.8)
        ax.axvline(int(CFG["fixation_yx"][1] * S), color="white", lw=0.8, alpha=0.8)
        ax.plot(int(CFG["fixation_yx"][1] * S), int(CFG["fixation_yx"][0] * S),
                "w+", markersize=9, markeredgewidth=1.8)
        ax.axis("off")
        if row_i == 0:
            ax.set_title(col_headers2[col_j], fontsize=8, pad=4)
        if row_i == n_rows - 1:
            ax.set_xlabel(f"SNR(corner)={snr_dbi:.1f} dB", fontsize=7, labelpad=2)

plt.tight_layout()
path_sweep = os.path.join(CFG["result_dir"], "02_periphery_beta_sweep.png")
common.save_figure(fig2, path_sweep, dpi=130)
plt.close(fig2)
print(f"Saved: {path_sweep}")

print(f"\nSNR(corner={r_corner:.2f}°) per β:")
for bv in beta_vals:
    p = SNRProfile(snr0_db=CFG["snr0_db"], beta=bv, ppd=CFG["ppd"])
    snr_dbi = 10 * np.log10(float(p.snr(torch.tensor(r_corner)).item()))
    alpha_i  = float(p.alpha(torch.tensor(r_corner)).item())
    print(f"  β={bv:.0f}: SNR={snr_dbi:.1f} dB  →  α(corner)={alpha_i:.3f}  "
          f"({'mostly signal' if alpha_i > 0.7 else 'mostly texture' if alpha_i < 0.4 else 'mixed'})")


In [ ]:
# ── SNR profile plot ─────────────────────────────────────────────────────
# Show SNR(r) and alpha(r) for beta sweep, plus Watson s(r) on secondary axis.

r_plot = torch.linspace(0.01, 8.0, 200)
watson_plot = WatsonPoolingScale()
s_r = np.array([float(watson_plot.spacing(r.item())) for r in r_plot]) * 60  # arcmin

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# --- Panel A: SNR(r) for beta sweep ---
ax = axes[0]
for beta_val in CFG["beta_sweep"]:
    prof_tmp = SNRProfile(snr0_db=CFG["snr0_db"], beta=beta_val, ppd=CFG["ppd"])
    snr_db_vals = np.array([10 * np.log10(float(prof_tmp.snr(torch.tensor(r.item())).item()))
                             for r in r_plot])
    ax.plot(r_plot.numpy(), snr_db_vals, label=f"β={beta_val}")
ax.set_xlabel("Eccentricity r [deg]")
ax.set_ylabel("SNR [dB]")
ax.set_title("SNR(r) profile — β sweep")
ax.legend(fontsize=8)
ax.axhline(0, color="gray", lw=0.7, ls="--")
# Mark image edges for CIFAR-10 and ImageNet
max_ecc_cifar = float(eccentricity_map(32, 32)[1].max().item()) * 32 / 32  # ~5.7 at ppd=4
max_ecc_in = float(eccentricity_map(224, 224, ppd=28.0)[1].max().item())
ax.axvline(max_ecc_cifar * (4.0 / CFG["ppd"]), color="C0", lw=1, ls=":", alpha=0.7,
           label="CIFAR-10 edge")
ax.grid(True, alpha=0.3)

# --- Panel B: Watson spacing s(r) ---
ax = axes[1]
ax.plot(r_plot.numpy(), s_r, "k-", lw=2)
ax.set_xlabel("Eccentricity r [deg]")
ax.set_ylabel("mRGC spacing s(r) [arcmin]")
ax.set_title("Watson (2014) mRGC spacing")
ax.annotate(f"s(0)={float(watson_plot.spacing_fovea()*60):.2f} arcmin",
            xy=(0, float(watson_plot.spacing_fovea()*60)),
            xytext=(1.5, float(watson_plot.spacing_fovea()*60)+0.3),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=8)
ax.grid(True, alpha=0.3)

# --- Panel C: alpha(r) for beta sweep ---
ax = axes[2]
for beta_val in CFG["beta_sweep"]:
    prof_tmp = SNRProfile(snr0_db=CFG["snr0_db"], beta=beta_val, ppd=CFG["ppd"],
                           rho=CFG["rho"])
    alpha_vals = np.array([float(prof_tmp.alpha(torch.tensor(r.item())).item())
                            for r in r_plot])
    ax.plot(r_plot.numpy(), alpha_vals, label=f"β={beta_val}")
ax.set_xlabel("Eccentricity r [deg]")
ax.set_ylabel("Content fraction α(r)")
ax.set_title("alpha(r) — content/texture interpolation knob")
ax.legend(fontsize=8)
ax.axhline(0.5, color="gray", lw=0.7, ls="--", label="α=0.5")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

fig.suptitle("Watson (2014) pooling scale and SNR(r) profile", fontsize=11)
plt.tight_layout()
snr_path = os.path.join(CFG["result_dir"], "02_snr_profile.png")
common.save_figure(fig, snr_path, dpi=130)
plt.close(fig)
print(f"Saved: {snr_path}")


In [ ]:
# ── Equal-SNR budget validation ──────────────────────────────────────────
# For each periphery regime, measure empirical SNR at several eccentricity bands.
# This confirms A2–A4 are indeed at comparable SNR before training.

def measure_snr_map(original, corrupted):
    """Per-pixel empirical SNR across a batch of images.
    signal = original, noise = corrupted - original.
    Returns mean SNR (linear) over all pixels."""
    diff = (corrupted - original)
    signal_power = original.pow(2).mean()
    noise_power  = diff.pow(2).mean().clamp(min=1e-10)
    return (signal_power / noise_power).item()


# Build a small batch of random test images
common.set_seed(CFG["seed"])
test_imgs = torch.stack([make_synthetic_sample(S, S, seed=k) for k in range(20)])

_, r_deg_all = eccentricity_map(S, S, CFG["fixation_yx"], CFG["ppd"])
max_r_deg = float(r_deg_all.max().item())
n_bins = 4
bin_edges = torch.linspace(0, max_r_deg, n_bins + 1)

snr_prof_eval = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                            ppd=CFG["ppd"], rho=CFG["rho"])

results_snr = {}
for mode in ["none", "blur", "iid", "trace"]:
    ft = build_foveated_transform(
        mode, snr_prof_eval, CFG["patch_size"],
        CFG["fovea_deg"], CFG["transition_deg"], CFG["ppd"], CFG["fixation_yx"],
        CFG["rblur_sigma0"], CFG["rblur_slope"])
    bin_snrs = []
    with torch.no_grad():
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            mask_bin = (r_deg_all >= lo.item()) & (r_deg_all < hi.item())  # [H,W]
            if mask_bin.sum() == 0:
                continue
            orig_bin = test_imgs[:, :, mask_bin]   # [B, C, n_pixels]
            outs = torch.stack([ft(img) for img in test_imgs])  # [B, C, H, W]
            out_bin = outs[:, :, mask_bin]
            snr_lin = measure_snr_map(orig_bin, out_bin)
            bin_snrs.append((float((lo+hi).item()/2), 10*np.log10(max(snr_lin, 1e-6))))
    results_snr[mode] = bin_snrs

print(f"{'Mode':<8}  {'r [deg]':>8}  {'SNR_dB':>8}")
for mode, bins in results_snr.items():
    for r_mid, snr_db in bins:
        print(f"{mode:<8}  {r_mid:8.2f}  {snr_db:8.2f}")
    print()


## Write `src/foveation.py`

The cell below writes the production module `src/foveation.py` which mirrors all
the classes and functions defined above. Subsequent cells (and later notebooks)
import from this module.


In [ ]:
# Write src/foveation.py from the content defined in this notebook
_foveation_content = '''"""
Foveation transforms for the foveated-vision thesis.

Public API
----------
WatsonPoolingScale   -- mRGC receptive-field spacing s(r) [Watson 2014, J.Vis 14(7):15]
SNRProfile           -- SNR(r) = SNR0 * (s0/s(r))^beta with alpha(r) knob
eccentricity_map     -- pixel-distance eccentricity tensor (pixels + degrees)
gaussian_blur_2d     -- isotropic Gaussian convolution (depthwise, reflect pad)
soft_foveal_mask     -- sigmoid soft-mask: ~1 inside fovea, ~0 in periphery
apply_rblur          -- R-Blur: eccentricity-dependent Gaussian blur (pyramid blend)
FlatBlurPeriphery    -- Ablation A2: low-pass Gaussian in periphery
IIDNoisePeriphery    -- Ablation A3: signal-independent Gaussian noise
TraceBasedPeriphery  -- Ablation A4 (original): signal-conditioned AdaIN metameric
FoveatedTransform    -- Composite: sharp fovea mask + chosen periphery module

References
----------
[9]  Narang & Bhatt 2020 (R-Blur / RetinalBlur)
[23] Watson 2014, J. Vision 14(7):15  (mRGC spacing formula)
TRACE_BASED_NOISE_REHBERI.md  (SNR formalisation, original contribution)
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------------------------------------------------
# Watson (2014) mRGC pooling scale
# -----------------------------------------------------------------------

class WatsonPoolingScale:
    """
    Midget RGC receptive-field spacing as a function of eccentricity.

    Watson (2014) fits Curcio & Allen (1990) ganglion-cell density data and
    derives a near-linear relation between eccentricity r (degrees) and the
    mean mRGC inter-cell spacing s(r) (degrees):

        60 * s(r) ≈ 0.53 + 0.434 * r
    =>  s(r) = (0.53 + 0.434 * r) / 60   [degrees]

    At the fovea (r=0): s(0) = 0.53/60 ≈ 0.0088° ≈ 0.53 arcmin ≈ cone spacing.
    Pooling window radius: w(r) = kappa * s(r), kappa ≥ 1.
    """

    A = 0.434   # slope  [arcmin / degree]  (Watson 2014, Table 1 fit)
    B = 0.53    # intercept [arcmin] at r=0

    def spacing(self, r_deg):
        """s(r) in degrees. r_deg may be float, ndarray, or torch.Tensor."""
        return (self.B + self.A * r_deg) / 60.0

    def spacing_fovea(self):
        """s(0) in degrees — reference spacing at the fixation point."""
        return self.B / 60.0

    def window_radius(self, r_deg, kappa=1.0):
        """Pooling window radius w(r) = kappa * s(r) in degrees."""
        return kappa * self.spacing(r_deg)


# -----------------------------------------------------------------------
# SNR profile
# -----------------------------------------------------------------------

class SNRProfile:
    """
    Eccentricity-dependent SNR(r) derived from the Watson pooling scale.

    Formula (TRACE_BASED_NOISE_REHBERI.md Sec. 3.2):

        SNR(r) = SNR0 * (s(0) / s(r))^beta

    where s(r) is the Watson mRGC spacing, SNR0 is the foveal (r=0) SNR, and
    beta controls the roll-off steepness (beta=2 corresponds to area-based pooling).

    Content-texture interpolation parameter alpha(r) (Sec. 3.3):

        alpha(r) = sqrt(SNR(r)/rho) / (1 + sqrt(SNR(r)/rho))

    where rho is the signal-to-texture power ratio (default 1.0).
    """

    def __init__(self, snr0_db: float = 30.0, beta: float = 2.0,
                 ppd: float = 28.0, rho: float = 1.0):
        self.snr0 = 10.0 ** (snr0_db / 10.0)   # linear SNR at fovea
        self.beta = beta
        self.ppd = ppd
        self.rho = rho
        self._watson = WatsonPoolingScale()

    def snr(self, r_deg):
        """Linear SNR(r). r_deg may be float or torch.Tensor."""
        s0 = self._watson.spacing_fovea()
        sr = self._watson.spacing(r_deg)
        return self.snr0 * (s0 / sr) ** self.beta

    def snr_db(self, r_deg):
        """SNR(r) in decibels."""
        return 10.0 * torch.log10(self.snr(r_deg))

    def alpha(self, r_deg):
        """
        Content-texture interpolation coefficient alpha(r) in [0, 1].
        alpha -> 1 at fovea (full signal); alpha -> 0 far in periphery (pure texture).
        """
        snr_r = self.snr(r_deg)
        sqrt_term = torch.sqrt(snr_r / self.rho)
        return sqrt_term / (1.0 + sqrt_term)


# -----------------------------------------------------------------------
# Core geometry helpers
# -----------------------------------------------------------------------

def eccentricity_map(H: int, W: int, fixation_yx=(0.5, 0.5), ppd: float = 28.0):
    """
    Eccentricity tensor for an H x W image.

    Args:
        H, W:         image height and width in pixels.
        fixation_yx:  relative fixation (fy, fx) in [0, 1] x [0, 1].
        ppd:          pixels per degree of visual angle.

    Returns:
        r_pix: [H, W] eccentricity in pixels.
        r_deg: [H, W] eccentricity in degrees.
    """
    cy = fixation_yx[0] * H
    cx = fixation_yx[1] * W
    ys = torch.arange(H, dtype=torch.float32)
    xs = torch.arange(W, dtype=torch.float32)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    r_pix = torch.sqrt((gy - cy) ** 2 + (gx - cx) ** 2)
    r_deg = r_pix / ppd
    return r_pix, r_deg


def gaussian_blur_2d(image: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Isotropic Gaussian blur on a [C, H, W] or [B, C, H, W] tensor.
    Uses depthwise convolution with reflect padding.
    Returns the input unchanged if sigma < 1e-3.
    """
    if sigma < 1e-3:
        return image
    squeeze = image.dim() == 3
    if squeeze:
        image = image.unsqueeze(0)
    B, C, H, W = image.shape
    radius = max(1, min(int(math.ceil(3.0 * sigma)), min(H, W) - 1))
    ks = 2 * radius + 1
    xs = torch.arange(-radius, radius + 1, dtype=image.dtype, device=image.device)
    k1d = torch.exp(-0.5 * (xs / sigma) ** 2)
    k1d = k1d / k1d.sum()
    k2d = k1d[:, None] * k1d[None, :]  # [ks, ks]
    k2d = k2d.view(1, 1, ks, ks).expand(C, 1, ks, ks).contiguous()
    padded = F.pad(image, [radius] * 4, mode="replicate")
    blurred = F.conv2d(padded, k2d, groups=C)
    if squeeze:
        blurred = blurred.squeeze(0)
    return blurred


def soft_foveal_mask(H: int, W: int, fixation_yx=(0.5, 0.5),
                      fovea_deg: float = 1.5, transition_deg: float = 0.5,
                      ppd: float = 28.0) -> torch.Tensor:
    """
    Soft foveal mask in [0, 1]: ~1 inside the foveal region, ~0 in periphery.
    Uses a sigmoid transition:
        m(r) = sigmoid( -(r - fovea_deg) / transition_deg )
    """
    _, r_deg = eccentricity_map(H, W, fixation_yx, ppd)
    return torch.sigmoid(-(r_deg - fovea_deg) / transition_deg)


# -----------------------------------------------------------------------
# R-Blur: eccentricity-dependent Gaussian blur
# -----------------------------------------------------------------------

def apply_rblur(image: torch.Tensor,
                fixation_yx=(0.5, 0.5),
                sigma0: float = 0.5,
                slope: float = 1.5,
                ppd: float = 28.0,
                n_levels: int = 5) -> torch.Tensor:
    """
    R-Blur foveal transform: eccentricity-dependent Gaussian blur.

    Blur sigma at each pixel: sigma(r) = sigma0 + slope * r_deg  [pixels].
    Implemented via Gaussian pyramid blending with triangular window weights:
      1. Build N blurred versions at uniformly-spaced sigma levels.
      2. At each pixel, blend the two adjacent levels proportionally.

    Args:
        image:       [C, H, W] float tensor in [0, 1].
        fixation_yx: relative fixation (fy, fx) in [0, 1]^2.
        sigma0:      blur sigma at the fixation point (pixels).
        slope:       rate of sigma increase per degree of eccentricity (px/deg).
        ppd:         pixels per degree of visual angle.
        n_levels:    number of Gaussian pyramid levels.

    Returns: [C, H, W] float tensor in [0, 1].
    """
    C, H, W = image.shape
    device = image.device
    _, r_deg = eccentricity_map(H, W, fixation_yx, ppd)
    r_deg = r_deg.to(device)
    sigma_map = (sigma0 + slope * r_deg).clamp(min=0.0)  # [H, W]
    sigma_max = float(sigma_map.max().item())
    sigma_levels = torch.linspace(0.0, sigma_max, n_levels, device=device)
    # Build pyramid
    blurred_stack = torch.stack(
        [gaussian_blur_2d(image, float(s.item())) for s in sigma_levels], dim=0
    )  # [N, C, H, W]
    # Triangular blending weights at each pixel
    level_span = sigma_levels[1] - sigma_levels[0] + 1e-8  # uniform spacing
    dist = (sigma_map[None, :, :] - sigma_levels[:, None, None]).abs() / level_span
    weights = F.relu(1.0 - dist)           # [N, H, W]
    weights = weights / (weights.sum(0, keepdim=True) + 1e-8)
    output = (weights[:, None, :, :] * blurred_stack).sum(0)  # [C, H, W]
    return output.clamp(0.0, 1.0)


# -----------------------------------------------------------------------
# Periphery regime modules (A2, A3, A4)
# -----------------------------------------------------------------------

class FlatBlurPeriphery(nn.Module):
    """
    Ablation A2: Low-pass Gaussian blur in the periphery (no added noise).
    Implements R-Blur with a potentially larger slope to match SNR budget.
    """

    def __init__(self, sigma0: float = 0.5, slope: float = 2.0,
                 ppd: float = 28.0, fixation_yx=(0.5, 0.5)):
        super().__init__()
        self.sigma0 = sigma0
        self.slope = slope
        self.ppd = ppd
        self.fixation_yx = fixation_yx

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        return apply_rblur(image, self.fixation_yx, self.sigma0,
                           self.slope, self.ppd)


class IIDNoisePeriphery(nn.Module):
    """
    Ablation A3: Signal-independent i.i.d. Gaussian noise in the periphery.

    Noise amplitude is derived from the SNR profile so that at each eccentricity r
    the added noise power matches the target SNR(r):

        sigma_noise(r) = sqrt( Var[I] / SNR(r) )

    This makes A3 directly comparable to A4 (trace-based) at equal SNR budget.
    """

    def __init__(self, snr_profile: SNRProfile,
                 fixation_yx=(0.5, 0.5), ppd: float = 28.0):
        super().__init__()
        self.snr_profile = snr_profile
        self.fixation_yx = fixation_yx
        self.ppd = ppd

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        C, H, W = image.shape
        device = image.device
        _, r_deg = eccentricity_map(H, W, self.fixation_yx, self.ppd)
        r_deg = r_deg.to(device)
        snr_map = self.snr_profile.snr(r_deg).clamp(min=1e-3)  # [H, W]
        signal_var = image.var().clamp(min=1e-6)
        sigma_map = torch.sqrt(signal_var / snr_map)   # [H, W]
        noise = torch.randn_like(image)
        return (image + sigma_map[None] * noise).clamp(0.0, 1.0)


class TraceBasedPeriphery(nn.Module):
    """
    Ablation A4 — Original contribution: signal-conditioned AdaIN metameric noise.

    Peripheral output at pixel p with eccentricity r:

        I_hat(p) = alpha(r) * I(p) + (1 - alpha(r)) * T(S_{Omega(r)}(I))(p)

    where:
      * S_{Omega} = local AdaIN statistics (mean, std) over a patch of size
        proportional to the Watson pooling window (kappa * s(r) * ppd pixels).
      * T(S) = AdaIN texture sample: Gaussian noise rescaled to match local
        mean and std — this preserves the *trace* (local statistics) while
        randomising the fine spatial structure (phase).
      * alpha(r) from SNRProfile.alpha() — encodes Watson-grounded SNR decay.

    This is differentiable and can be used end-to-end in training.
    For adversarial evaluation, fix seed and use EOT-N.
    """

    def __init__(self, snr_profile: SNRProfile,
                 patch_size: int = 8,
                 fixation_yx=(0.5, 0.5),
                 ppd: float = 28.0):
        super().__init__()
        self.snr_profile = snr_profile
        self.patch_size = patch_size
        self.fixation_yx = fixation_yx
        self.ppd = ppd

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        C, H, W = image.shape
        device = image.device
        _, r_deg = eccentricity_map(H, W, self.fixation_yx, self.ppd)
        r_deg = r_deg.to(device)
        alpha_map = self.snr_profile.alpha(r_deg)         # [H, W]
        texture = self._adain_texture(image)               # [C, H, W]
        return (alpha_map[None] * image
                + (1.0 - alpha_map[None]) * texture).clamp(0.0, 1.0)

    def _adain_texture(self, image: torch.Tensor) -> torch.Tensor:
        """
        Generate an AdaIN texture sample matching local patch statistics.
        Steps:
          1. Pad image to next multiple of patch_size.
          2. Extract non-overlapping patches [C, nH, nW, p, p].
          3. Compute patch-level (mean, std).
          4. Normalise a random Gaussian draw, then rescale/shift to match.
          5. Reconstruct and crop to original size.
        """
        C, H, W = image.shape
        device = image.device
        p = self.patch_size
        pad_h = (p - H % p) % p
        pad_w = (p - W % p) % p
        img_p = F.pad(image, [0, pad_w, 0, pad_h], mode="reflect")  # [C, Hp, Wp]
        noise_p = torch.randn(C, H + pad_h, W + pad_w,
                              device=device, dtype=image.dtype)
        # Extract patches: [C, nH, nW, p, p]
        img_unf = img_p.unfold(1, p, p).unfold(2, p, p)
        noise_unf = noise_p.unfold(1, p, p).unfold(2, p, p)
        mu = img_unf.mean(dim=(-2, -1), keepdim=True)
        sigma = img_unf.std(dim=(-2, -1), keepdim=True).clamp(min=1e-5)
        noise_mu = noise_unf.mean(dim=(-2, -1), keepdim=True)
        noise_std = noise_unf.std(dim=(-2, -1), keepdim=True).clamp(min=1e-5)
        adain_unf = sigma * (noise_unf - noise_mu) / noise_std + mu
        # Reconstruct [C, Hp, Wp] from [C, nH, nW, p, p]
        nH = (H + pad_h) // p
        nW = (W + pad_w) // p
        texture = adain_unf.permute(0, 1, 3, 2, 4).reshape(C, nH * p, nW * p)
        return texture[:, :H, :W].clamp(0.0, 1.0)


# -----------------------------------------------------------------------
# Composite foveated transform
# -----------------------------------------------------------------------

class FoveatedTransform(nn.Module):
    """
    Composite foveated image transform.

    Output = m(r) * I + (1 - m(r)) * Periphery(I)

    where m(r) is the soft foveal mask (≈1 at fixation, ≈0 far periphery).
    The foveal region always sees the sharp, unmodified input image.

    Args:
        periphery:      nn.Module from {FlatBlurPeriphery, IIDNoisePeriphery,
                        TraceBasedPeriphery}, or None (no-op, A1 baseline).
        fovea_deg:      fovea radius in degrees.
        transition_deg: soft-mask transition width in degrees.
        ppd:            pixels per degree.
        fixation_yx:    relative fixation (fy, fx) in [0,1]^2.
    """

    def __init__(self, periphery: nn.Module = None,
                 fovea_deg: float = 1.5,
                 transition_deg: float = 0.5,
                 ppd: float = 28.0,
                 fixation_yx=(0.5, 0.5)):
        super().__init__()
        self.periphery = periphery
        self.fovea_deg = fovea_deg
        self.transition_deg = transition_deg
        self.ppd = ppd
        self.fixation_yx = fixation_yx

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        C, H, W = image.shape
        device = image.device
        mask = soft_foveal_mask(H, W, self.fixation_yx,
                                self.fovea_deg, self.transition_deg,
                                self.ppd).to(device)   # [H, W]
        if self.periphery is None:
            return image
        periph = self.periphery(image)  # [C, H, W]
        return (mask[None] * image + (1.0 - mask[None]) * periph).clamp(0.0, 1.0)


def build_foveated_transform(mode: str, snr_profile: SNRProfile,
                              patch_size: int = 8,
                              fovea_deg: float = 1.5,
                              transition_deg: float = 0.5,
                              ppd: float = 28.0,
                              fixation_yx=(0.5, 0.5),
                              blur_sigma0: float = 0.5,
                              blur_slope: float = 2.0) -> FoveatedTransform:
    """
    Factory: build a FoveatedTransform for ablation mode in
    {'none', 'rblur', 'blur', 'iid', 'trace'}.
    """
    if mode == "none":
        periphery = None
    elif mode == "rblur":
        periphery = FlatBlurPeriphery(blur_sigma0, blur_slope, ppd, fixation_yx)
    elif mode == "blur":
        periphery = FlatBlurPeriphery(blur_sigma0, blur_slope, ppd, fixation_yx)
    elif mode == "iid":
        periphery = IIDNoisePeriphery(snr_profile, fixation_yx, ppd)
    elif mode == "trace":
        periphery = TraceBasedPeriphery(snr_profile, patch_size, fixation_yx, ppd)
    else:
        raise ValueError(f"Unknown periphery mode: {mode!r}")
    return FoveatedTransform(periphery, fovea_deg, transition_deg, ppd, fixation_yx)'''

_fov_path = os.path.join(PROJECT_ROOT, 'src', 'foveation.py')
with open(_fov_path, 'w') as _f:
    _f.write(_foveation_content)
print(f'Written: {_fov_path}')

# Reload module if it was already imported
import importlib
if 'src.foveation' in sys.modules:
    importlib.reload(sys.modules['src.foveation'])
from src import foveation as fov_module
print('src.foveation imported OK')

# Verify key public symbols are accessible
_expected = ['WatsonPoolingScale', 'SNRProfile', 'eccentricity_map',
             'gaussian_blur_2d', 'soft_foveal_mask', 'apply_rblur',
             'FlatBlurPeriphery', 'IIDNoisePeriphery', 'TraceBasedPeriphery',
             'FoveatedTransform', 'build_foveated_transform']
for _sym in _expected:
    assert hasattr(fov_module, _sym), f'Missing: {_sym}'
print(f'Verified {len(_expected)} public symbols in src.foveation')

## Ablation Smoke Test: Clean Accuracy vs. Periphery Mode

Train a tiny classifier on foveated CIFAR-10 images for each periphery mode
(`none` / `blur` / `iid` / `trace`). In smoke-test mode this runs for a handful of
epochs on synthetic data — enough to verify the pipeline is end-to-end differentiable
and produces sensible gradients.


In [ ]:
# ── Tiny classifier definition ────────────────────────────────────────────

class TinyCNN(nn.Module):
    """Small CNN for quick ablation sweeps. AdaptiveAvgPool2d(4) before the
    classifier makes it resolution-agnostic (works unchanged at CIFAR-10's 32x32
    and ImageNet-100's 224x224)."""
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),    # always 4x4, regardless of input resolution
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── Real data loaders: CIFAR-10 (mandatory) + ImageNet-100 (efficiency comparison) ──
# Per-dataset geometry: CIFAR-10 at native 32x32 (ppd=4, matching CFG's fovea/blur
# defaults above); ImageNet-100 at 224x224 with VOneNet's ppd=28 convention (matches
# the `if CFG["dataset"] == "imagenet": ...` override block above).
DATASET_GEOMETRY = {
    "cifar10":     dict(image_size=32,  ppd=4.0,  fovea_deg=1.0, rblur_slope=1.5,
                         patch_size=8,  n_classes=10),
    "imagenet100": dict(image_size=224, ppd=28.0, fovea_deg=1.5, rblur_slope=2.0,
                         patch_size=32, n_classes=100),
}


def build_real_loaders(dataset_name, batch_size=64):
    """Return (train_loader, val_loader) for `dataset_name` in {'cifar10', 'imagenet100'}.

    Uses `src/datasets.py`'s honest-fallback loaders:
      * CIFAR-10: auto-downloads (~170 MB) when `smoke_test=False`.
      * ImageNet-100: license-gated, not auto-downloaded -- must already be placed
        under `data/imagenet100/{train,val}/<class>/*.JPEG`. Raises FileNotFoundError
        with the expected layout if missing and `smoke_test=False` (never fabricates
        data); falls back to a clearly-labelled synthetic pipeline check only when
        `smoke_test=True`.
    """
    if dataset_name == "cifar10":
        train_ds = datasets.get_cifar10(CFG["data_dir"], train=True, normalization="imagenet",
                                         download=True, smoke_test=CFG["smoke_test"])
        val_ds   = datasets.get_cifar10(CFG["data_dir"], train=False, normalization="imagenet",
                                         download=True, smoke_test=CFG["smoke_test"])
    elif dataset_name == "imagenet100":
        geom = DATASET_GEOMETRY["imagenet100"]
        train_ds = datasets.get_imagenet100(CFG["data_dir"], split="train", normalization="imagenet",
                                             image_size=geom["image_size"], smoke_test=CFG["smoke_test"])
        val_ds   = datasets.get_imagenet100(CFG["data_dir"], split="val", normalization="imagenet",
                                             image_size=geom["image_size"], smoke_test=CFG["smoke_test"])
    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name!r}")

    num_workers = 2 if not CFG["smoke_test"] else 0
    train_ld = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                            num_workers=num_workers)
    val_ld   = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                                            num_workers=num_workers)
    return train_ld, val_ld


# ── Training & evaluation helpers ─────────────────────────────────────────

def train_one_epoch(model, fov_transform, loader, optimiser, device,
                     n_batches_max=None, normalisation_mean=None, normalisation_std=None):
    """One training epoch with foveal transform applied before the classifier.
    The fov_transform operates on raw [0,1] pixels; the normalised batch comes
    from the DataLoader, so we undo normalisation, apply fovea, re-normalise."""
    model.train()
    total_loss = correct = total = 0
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    for batch_i, (xb, yb) in enumerate(loader):
        if n_batches_max is not None and batch_i >= n_batches_max:
            break
        xb, yb = xb.to(device), yb.to(device)
        # Undo normalisation to get raw [0,1] pixels for foveal transform
        raw = (xb * std + mean).clamp(0.0, 1.0)
        # Apply foveal transform per-sample (differentiable)
        fov_list = []
        for i in range(raw.size(0)):
            fov_list.append(fov_transform(raw[i]))
        fov_batch = torch.stack(fov_list)           # [B, C, H, W]
        # Re-normalise
        xb_fov = (fov_batch - mean) / std
        logits = model(xb_fov)
        loss = F.cross_entropy(logits, yb)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def eval_accuracy(model, fov_transform, loader, device, n_batches_max=None):
    """Evaluate top-1 accuracy with foveal transform applied."""
    model.eval()
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    correct = total = 0
    for batch_i, (xb, yb) in enumerate(loader):
        if n_batches_max is not None and batch_i >= n_batches_max:
            break
        xb, yb = xb.to(device), yb.to(device)
        raw = (xb * std + mean).clamp(0.0, 1.0)
        fov_list = [fov_transform(raw[i]) for i in range(raw.size(0))]
        fov_batch = torch.stack(fov_list)
        xb_fov = (fov_batch - mean) / std
        logits = model(xb_fov)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return correct / max(total, 1)

In [ ]:
# ── Ablation loop: CIFAR-10 (mandatory) + ImageNet-100 (efficiency comparison) ──

common.set_seed(CFG["seed"])

n_epochs  = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]
n_batches_val = 10 if CFG["smoke_test"] else None

ablation_results = {}

for dataset_name in CFG["eval_datasets"]:
    geom = DATASET_GEOMETRY[dataset_name]
    print(f"\n{'='*70}\nDataset: {dataset_name}  "
          f"(image_size={geom['image_size']}, n_classes={geom['n_classes']})\n{'='*70}")

    try:
        train_ld, val_ld = build_real_loaders(dataset_name, CFG["batch_size"])
    except FileNotFoundError as e:
        print(f"  Skipping {dataset_name}: {e}")
        continue

    snr_prof_abl = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"],
                               ppd=geom["ppd"], rho=CFG["rho"])

    for mode in CFG["ablation_modes"]:
        print(f"\n── [{dataset_name}] Ablation mode: {mode} ──────────────────────")
        common.set_seed(CFG["seed"])
        model = TinyCNN(n_classes=geom["n_classes"]).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
        ft    = build_foveated_transform(
            mode, snr_prof_abl, geom["patch_size"],
            geom["fovea_deg"], CFG["transition_deg"], geom["ppd"], CFG["fixation_yx"],
            CFG["rblur_sigma0"], geom["rblur_slope"])

        history = []
        for epoch in range(n_epochs):
            tr_loss, tr_acc = train_one_epoch(
                model, ft, train_ld, opt, device, n_batches_max=n_batches)
            val_acc = eval_accuracy(model, ft, val_ld, device, n_batches_max=n_batches_val)
            history.append({"epoch": epoch, "train_loss": tr_loss,
                            "train_acc": tr_acc, "val_acc": val_acc})
            print(f"  Epoch {epoch+1}/{n_epochs}  "
                  f"loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}")

        ablation_results[f"{dataset_name}_{mode}"] = {
            "dataset": dataset_name,
            "mode": mode,
            "final_val_acc": val_acc,
            "history": history,
            "note": ("smoke_test=True: real data if present, else a clearly-labelled "
                     "synthetic pipeline check (see datasets.py honest-fallback policy)"
                     if CFG["smoke_test"] else "Full run: real data."),
        }

print("\n─── Ablation summary ───────────────────────────────────────")
print(f"{'Dataset':<14}{'Mode':<8}  {'Final val acc':>14}")
for key, r in ablation_results.items():
    print(f"{r['dataset']:<14}{r['mode']:<8}  {r['final_val_acc']:14.4f}")
if CFG["smoke_test"]:
    print("\n(smoke_test=True: numbers reflect the pipeline, not final results -- "
          "set CFG['smoke_test']=False for real CIFAR-10 numbers; ImageNet-100 "
          "additionally requires data/imagenet100/ to be populated locally.)")

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────

output = {
    "notebook": "02_foveation_rblur_and_periphery",
    "cfg": {k: v for k, v in CFG.items()
            if not k.endswith("_dir")},   # exclude paths (non-serialisable)
    "ablation_results": ablation_results,
    "note": (
        "smoke_test=True: all accuracy numbers are from a randomly-initialised "
        "model on synthetic data. Run with smoke_test=False for real baselines."
        if CFG["smoke_test"] else "Full run results."
    ),
}

result_path = os.path.join(CFG["result_dir"], "02_metrics.json")
common.save_json(output, result_path)
common.save_config(CFG, os.path.join(CFG["result_dir"], "02_config.json"))
print(f"Results saved to: {result_path}")
print(f"Config  saved to: {os.path.join(CFG['result_dir'], '02_config.json')}")

# Summary
print("\n── Notebook 02 complete ──────────────────────────────────────────────")
print(f"  src/foveation.py    : written")
print(f"  02_periphery_samples.png : {os.path.exists(os.path.join(CFG['result_dir'], '02_periphery_samples.png'))}")
print(f"  02_snr_profile.png  : {os.path.exists(os.path.join(CFG['result_dir'], '02_snr_profile.png'))}")
print(f"  02_metrics.json     : {os.path.exists(result_path)}")
